# Practical 6 · From notebook to product (churn)

**Goal:** turn a trained model into something other people can use safely:
1. save the **whole pipeline** + a record of how it was made
2. load it in a **fresh session** and predict on **new rows** (batch inference)
3. **check inputs** before predicting
4. check **fairness** across groups
5. detect **drift**
6. write a **model card**
7. put it behind a **Streamlit app**

## Train the model we will ship (same steps as Practical 2)

In [1]:
import os
from urllib.request import urlretrieve
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_style('whitegrid'); plt.rcParams['figure.figsize'] = (8, 4.5)

# Step 1 - load
url = 'https://raw.githubusercontent.com/MusaBangash/Python-Variables/main/ml/data/telco-churn.csv'
if not os.path.exists('telco-churn.csv'):
    urlretrieve(url, 'telco-churn.csv')
df = pd.read_csv('telco-churn.csv')

# Step 4 - clean: blank TotalCharges belong to brand-new customers (tenure 0) -> they have paid 0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# Step 3 - target and features (drop the ID column)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']

# Step 6 - stratified split (same split as Practical 2)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Step 7 - feature types
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
categorical_cols = [c for c in X.columns if c not in numeric_cols]
print('Train', X_train.shape, '| Test', X_test.shape, '| churn rate', round(y_train.mean(), 3))

Train (5634, 19) | Test (1409, 19) | churn rate 0.265


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score, precision_score

RANDOM_STATE = 42                     # one seed, written down, used everywhere
preprocess = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_cols),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
])
model = Pipeline([('preprocess', preprocess), ('model', LogisticRegression(max_iter=2000))]).fit(X_train, y_train)
THRESHOLD = 0.25                      # chosen from business costs in Practical 2
test_proba = model.predict_proba(X_test)[:, 1]
print('Test ROC AUC:', round(roc_auc_score(y_test, test_proba), 3))

Test ROC AUC: 0.842


## 1 · Save the model and an experiment record
We save three things:
- **`churn_model.joblib`**: the whole pipeline (preprocessing + model)
- **`schema.json`**: which columns the model expects, their types and allowed values/ranges
- **`model_info.json`**: how it was made (data, seed, threshold, scores, library version), so anyone can reproduce or audit it

In [3]:
import joblib, json, sklearn, datetime

joblib.dump(model, 'churn_model.joblib')

schema = {}
for c in numeric_cols:
    schema[c] = {'type': 'numeric', 'min': float(X_train[c].min()), 'max': float(X_train[c].max())}
for c in categorical_cols:
    schema[c] = {'type': 'categorical', 'values': sorted(X_train[c].astype(str).unique().tolist())}
json.dump(schema, open('schema.json', 'w'), indent=2)

info = {'model': 'LogisticRegression in a preprocessing pipeline', 'target': 'Churn (1 = leaves)',
        'training_rows': len(X_train), 'test_rows': len(X_test), 'random_state': RANDOM_STATE,
        'threshold': THRESHOLD, 'test_roc_auc': round(roc_auc_score(y_test, test_proba), 4),
        'test_recall_at_threshold': round(recall_score(y_test, test_proba >= THRESHOLD), 4),
        'test_precision_at_threshold': round(precision_score(y_test, test_proba >= THRESHOLD), 4),
        'sklearn_version': sklearn.__version__, 'created': datetime.date.today().isoformat()}
json.dump(info, open('model_info.json', 'w'), indent=2)
print(json.dumps(info, indent=2))

{
  "model": "LogisticRegression in a preprocessing pipeline",
  "target": "Churn (1 = leaves)",
  "training_rows": 5634,
  "test_rows": 1409,
  "random_state": 42,
  "threshold": 0.25,
  "test_roc_auc": 0.8421,
  "test_recall_at_threshold": 0.8102,
  "test_precision_at_threshold": 0.4975,
  "sklearn_version": "1.9.1",
  "created": "2026-11-13"
}


## 2 · A fresh session: load and predict on new rows (batch inference)
In real life, prediction runs in another program, often days later. Below we **delete** everything from memory and start again with only the saved files. (In Colab you could also use *Runtime → Restart session* and run only this part.)

In [4]:
del model, preprocess          # forget the trained objects
import joblib, json
import pandas as pd

model = joblib.load('churn_model.joblib')
schema = json.load(open('schema.json'))
info = json.load(open('model_info.json'))

new_customers = X_test.head(8).copy()                 # pretend this is tomorrow's file
new_customers.to_csv('new_customers.csv', index=False)

batch = pd.read_csv('new_customers.csv')
batch['churn_probability'] = model.predict_proba(batch)[:, 1].round(3)
batch['call'] = batch['churn_probability'] >= info['threshold']
batch[['tenure', 'Contract', 'MonthlyCharges', 'churn_probability', 'call']]

,tenure,Contract,MonthlyCharges,churn_probability,call
0,72,Two year,114.05,0.046,False
1,8,Month-to-month,100.15,0.684,True
2,41,One year,78.35,0.059,False
3,18,Month-to-month,78.20,0.402,True
4,72,Two year,82.65,0.021,False
5,21,Month-to-month,99.85,0.603,True
6,21,Month-to-month,99.15,0.448,True
7,19,Month-to-month,24.10,0.130,False


The loaded pipeline does all preprocessing itself: we pass raw rows, exactly like the training file.

## 3 · Check inputs before predicting (input schema validation)
Models don't complain about nonsense; they happily predict for a customer with tenure −5 or contract "banana". We must check first:
- **errors** (don't predict): missing columns, text in a number column
- **warnings** (predict, but a human should look): values outside the training range, unseen categories, missing values

In [5]:
def validate(df, schema):
    errors, warnings = [], []
    for col, spec in schema.items():
        if col not in df.columns:
            errors.append(f'missing column: {col}'); continue
        if spec['type'] == 'numeric':
            values = pd.to_numeric(df[col], errors='coerce')
            if (values.isna() & df[col].notna()).any():
                errors.append(f'{col}: contains non-numbers')
            outside = (values < spec['min']) | (values > spec['max'])
            if outside.any():
                warnings.append(f"{col}: {int(outside.sum())} value(s) outside training range {spec['min']}–{spec['max']}")
        else:
            unseen = set(df[col].dropna().astype(str)) - set(spec['values'])
            if unseen:
                warnings.append(f'{col}: unseen categories {sorted(unseen)}')
    return errors, warnings

bad = batch.drop(columns=['churn_probability', 'call']).head(3).copy()
bad.loc[bad.index[0], 'tenure'] = -5
bad.loc[bad.index[1], 'Contract'] = 'Three year'
bad = bad.drop(columns=['PaymentMethod'])
errors, warnings = validate(bad, schema)
print('ERRORS  :', errors)
print('WARNINGS:', warnings)

ERRORS  : ['missing column: PaymentMethod']
WARNINGS: ['tenure: 1 value(s) outside training range 0.0–72.0', "Contract: unseen categories ['Three year']"]


Because of the **error** (a missing column), this file must be rejected, not predicted. The **warnings** would be logged for a person to review. In production this check runs before every prediction.

## 4 · Fairness: does the model work equally well for different groups?
The same metric, per group. Big gaps need an explanation in the model card. We check `gender` and `SeniorCitizen` on the test set.

In [6]:
test = X_test.assign(actual=y_test.values, flagged=(test_proba >= THRESHOLD).astype(int))
rows = []
for col in ['gender', 'SeniorCitizen']:
    for value, g in test.groupby(col):
        rows.append({'column': col, 'group': value, 'customers': len(g), 'actual churn rate': g['actual'].mean(),
                     'flagged rate': g['flagged'].mean(), 'recall': recall_score(g['actual'], g['flagged']),
                     'precision': precision_score(g['actual'], g['flagged'])})
fairness = pd.DataFrame(rows).round(3)
fairness

,column,group,customers,actual churn rate,flagged rate,recall,precision
0,gender,Female,687,0.281,0.431,0.793,0.517
1,gender,Male,722,0.251,0.434,0.829,0.479
2,SeniorCitizen,0,1187,0.233,0.382,0.764,0.465
3,SeniorCitizen,1,222,0.441,0.698,0.939,0.594


**Reading it:**
- **Gender:** men and women get similar recall (0.83 vs 0.79) and precision. No big gap.
- **Senior citizens:** flagged far more often (70% vs 38%), but they also **actually** leave far more often (44% vs 23%). The model catches **94%** of senior leavers but only **76%** of non-senior leavers: non-senior leavers are missed more often.

This is not automatically "unfair", but it is a real **difference in quality between groups**, and it must be written in the model card. The business should decide: is it acceptable that seniors get far more retention calls (the offer must be helpful, not pushy), and do we need a lower threshold for non-seniors to catch more of them?

**Should we just delete `gender` from the features?** Removing a sensitive column doesn't guarantee fairness (other columns can act as proxies), but if it adds nothing to accuracy, removing it is a sensible, simple choice. *(Exercise 1.)*

## 5 · Drift: is new data like the training data?
Models quietly get worse when the world changes. A standard check is the **Population Stability Index (PSI)** on the model's scores: below 0.1 stable, 0.1–0.25 watch, above 0.25 investigate.

In [7]:
def psi(expected, actual, bins=10):
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1)); edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected) + 1e-6
    a = np.histogram(actual, edges)[0] / len(actual) + 1e-6
    return float(np.sum((a - e) * np.log(a / e)))

train_scores = model.predict_proba(X_train)[:, 1]
print(f'PSI, this month (test set) vs training: {psi(train_scores, test_proba):.3f}')

future = X_test.copy()                                # simulate: a marketing push brings many brand-new customers
idx = future.sample(frac=0.5, random_state=1).index
future.loc[idx, 'tenure'] = 1
future.loc[idx, 'TotalCharges'] = future.loc[idx, 'MonthlyCharges']
future.loc[idx, 'Contract'] = 'Month-to-month'
print(f'PSI after a wave of new customers:      {psi(train_scores, model.predict_proba(future)[:, 1]):.3f}')

PSI, this month (test set) vs training: 0.014
PSI after a wave of new customers:      0.228


This month is stable (PSI ≈ 0.01). The simulated wave of new customers raises PSI to about **0.23**: the "watch" zone, close to the 0.25 alarm. A team would start investigating now.

- **Data drift:** the inputs change (above).
- **Performance drift:** when real outcomes arrive weeks later, compare actual recall/precision with the model card's numbers.
When either alarm rings: investigate, collect fresh labelled data, retrain with the same pipeline, test, and update the model card.

## 6 · The model card
A **model card** is a one-page description that travels with the model: what it's for, how good it is, where it fails, and its risks. We fill it from the numbers above.

In [8]:
fair = fairness.set_index(['column', 'group'])
card = f'''# Model card: customer churn risk

## Intended use
Rank current customers by probability of cancelling next month, so the retention team can call the riskiest first.
**Not for:** pricing, credit or any decision that removes a service from a customer.

## Data
IBM Telco customer sample: {info['training_rows']} training / {info['test_rows']} test customers, 19 account features.
Blank TotalCharges (11 brand-new customers) filled with 0. Customer ID removed.

## Model
{info['model']}. Threshold {info['threshold']} chosen from business costs (call $100, 30% of called leavers stay, customer worth $1,500).

## Performance (test set, opened once)
ROC AUC {info['test_roc_auc']} · at the threshold: recall {info['test_recall_at_threshold']}, precision {info['test_precision_at_threshold']}.

## Fairness check (test set)
Recall women {fair.loc[('gender','Female'),'recall']:.2f} vs men {fair.loc[('gender','Male'),'recall']:.2f}.
Seniors flagged {fair.loc[('SeniorCitizen',1),'flagged rate']:.0%} vs non-seniors {fair.loc[('SeniorCitizen',0),'flagged rate']:.0%};
their actual churn rate is also higher ({fair.loc[('SeniorCitizen',1),'actual churn rate']:.0%} vs {fair.loc[('SeniorCitizen',0),'actual churn rate']:.0%}).

## Limitations
- One snapshot of one company; patterns may not transfer to other markets or later years.
- Associations, not causes: "fiber customers leave more" does not say why.
- About half of flagged customers will not actually leave (precision ~0.5): the offer must be harmless to them.

## Risks and responsible use
- Personal data: store only what is needed; restrict access; delete when no longer needed.
- Monitor PSI on scores weekly; investigate above 0.25. Retrain when real outcomes show recall dropping.

## Reproducibility
random_state {info['random_state']}, scikit-learn {info['sklearn_version']}, created {info['created']}.
'''
open('MODEL_CARD.md', 'w').write(card)
print(card)

# Model card: customer churn risk

## Intended use
Rank current customers by probability of cancelling next month, so the retention team can call the riskiest first.
**Not for:** pricing, credit or any decision that removes a service from a customer.

## Data
IBM Telco customer sample: 5634 training / 1409 test customers, 19 account features.
Blank TotalCharges (11 brand-new customers) filled with 0. Customer ID removed.

## Model
LogisticRegression in a preprocessing pipeline. Threshold 0.25 chosen from business costs (call $100, 30% of called leavers stay, customer worth $1,500).

## Performance (test set, opened once)
ROC AUC 0.8421 · at the threshold: recall 0.8102, precision 0.4975.

## Fairness check (test set)
Recall women 0.79 vs men 0.83.
Seniors flagged 70% vs non-seniors 38%;
their actual churn rate is also higher (44% vs 23%).

## Limitations
- One snapshot of one company; patterns may not transfer to other markets or later years.
- Associations, not causes: "fiber customer

## 7 · A Streamlit app
**Streamlit** turns a Python script into a web page. The app below loads the saved model, shows a form, validates the input, and shows the churn probability and decision. `%%writefile` saves the cell as a file.

In [9]:
%%writefile app.py
import json
import joblib
import pandas as pd
import streamlit as st

st.set_page_config(page_title='Churn risk')
model = joblib.load('churn_model.joblib')
schema = json.load(open('schema.json'))
info = json.load(open('model_info.json'))

st.title('Customer churn risk')
st.caption(f"Logistic regression · test ROC AUC {info['test_roc_auc']} · threshold {info['threshold']}")

values = {}
for col, spec in schema.items():
    if spec['type'] == 'numeric':
        values[col] = st.number_input(col, min_value=spec['min'], max_value=spec['max'], value=spec['min'])
    else:
        values[col] = st.selectbox(col, spec['values'])

if st.button('Predict'):
    row = pd.DataFrame([values])
    p = float(model.predict_proba(row)[:, 1][0])
    st.metric('Probability of leaving', f'{p:.0%}')
    if p >= info['threshold']:
        st.error('High risk: add to the call list')
    else:
        st.success('Low risk')

with st.expander('Model card'):
    st.markdown(open('MODEL_CARD.md').read())

Overwriting app.py

### Test the app without a browser
Streamlit has a testing tool that runs the app in the background. This checks that it loads and predicts.

In [10]:
from streamlit.testing.v1 import AppTest
import os
app = AppTest.from_file(os.path.abspath('app.py'), default_timeout=60).run()
app.button[0].click().run()
print('Errors:', [e for e in app.exception])
print('Title:', app.title[0].value)
print('Result:', [(m.label, m.value) for m in app.metric])

2026-11-13 19:52:45.033 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Errors: []
Title: Customer churn risk
Result: [('Probability of leaving', '47%')]


No errors: the app loads the model, shows the form, predicts and displays the result. (A 'missing ScriptRunContext' warning is normal in this test mode.)

### Run it for real
- **On your laptop:** `pip install streamlit scikit-learn pandas joblib`, put `app.py`, `churn_model.joblib`, `schema.json`, `model_info.json` and `MODEL_CARD.md` in one folder, then run `streamlit run app.py`. A browser tab opens.
- **In Colab:** run `!streamlit run app.py &>/dev/null &` and then `!npx localtunnel --port 8501`, open the link it prints, and enter the password shown by `!curl ipv4.icanhazip.com`. (Tunnels change often; the laptop route is the most reliable.)

## When a model is NOT the right solution
Before shipping, ask: would a simple rule do almost as well (Session 1: "month-to-month → call" catches 88% of leavers)? Is the decision high-stakes with no human check? Are the labels reliable? Would people be harmed by false positives? If the answers worry you, ship a rule, keep a human in the loop, or don't ship.

## Exercises
1. Remove `gender` from the features, retrain, and compare ROC AUC and the fairness table. Would you keep it?
2. Add a check to `validate` that rejects rows where `TotalCharges` is much larger than `tenure × MonthlyCharges` (a likely data error).
3. Add a "Batch" section to the app: upload a CSV, validate it, and download predictions.